<a href="https://colab.research.google.com/github/pop123-ux/HuggingFace-Project-Learning/blob/main/course/chapter11/LoRA_using_SFT/LoRA%20SFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
pip install --upgrade torchao

# How to Fine-Tune LLMs with LoRA Adapters using Hugging Face TRL

This notebook demonstrates how to efficiently fine-tune large language models using LoRA (Low-Rank Adaptation) adapters. LoRA is a parameter-efficient fine-tuning technique that:
- Freezes the pre-trained model weights
- Adds small trainable rank decomposition matrices to attention layers
- Typically reduces trainable parameters by ~90%
- Maintains model performance while being memory efficient

We'll cover:
1. Setup development environment and LoRA configuration
2. Create and prepare the dataset for adapter training
3. Fine-tune using `trl` and `SFTTrainer` with LoRA adapters
4. Test the model and merge adapters (optional)


## Loading LoRA Adapters with PEFT ##

PEFT is a library that provides a unified interface for loading and managing PEFT methods, including LoRA. It allows to easily load and switch between different PEFT methods, making it easier to experiment with different fine-tuning techniques.

Adapters can be loaded onto a pretrained model with load_adapter(), which is useful for trying out different adapters whose weights aren't merged. Set the active adapter weights with the set_adapter() function. To return the base model, we could use unload() to unload all the LoRA modules. This makes it easy to switch between different task-specific weights.


In [1]:
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM

config = PeftConfig.from_pretrained('ybelkada/opt-350m-lora')
model = AutoModelForCausalLM.from_pretrained(config.base_model_name_or_path)
model = PeftModel.from_pretrained(model, 'ybelkada/opt-350m-lora')

Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

In [9]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): OPTForCausalLM(
      (model): OPTModel(
        (decoder): OPTDecoder(
          (embed_tokens): Embedding(50272, 512, padding_idx=1)
          (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
          (project_out): Linear(in_features=1024, out_features=512, bias=False)
          (project_in): Linear(in_features=512, out_features=1024, bias=False)
          (layers): ModuleList(
            (0-23): 24 x OPTDecoderLayer(
              (self_attn): OPTAttention(
                (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (v_proj): lora.Linear(
                  (base_layer): Linear(in_features=1024, out_features=1024, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.05, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=1024, out_features=16, bias=False

In [ ]:
!pip install --upgrade transformers sentencepiece
import os
os._exit(0)


In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('facebook/opt-350m')
print(tokenizer)


tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

GPT2Tokenizer(name_or_path='facebook/opt-350m', vocab_size=50265, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '</s>', 'unk_token': '</s>', 'pad_token': '<pad>'}, added_tokens_decoder={
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
})


## 1. Setup development environment

Our first step is to install Hugging Face Libraries and Pytorch, including trl, transformers and datasets. If you haven't heard of trl yet, don't worry. It is a new library on top of transformers and datasets, which makes it easier to fine-tune, rlhf, align open LLMs.


In [3]:
# Install the requirements in Google Colab
!pip install transformers datasets trl huggingface_hub

# Authenticate to Hugging Face

from huggingface_hub import login

login()

# for convenience you can create an environment variable containing your hub token as HF_TOKEN

## 2. Load the dataset

In [6]:
# Load a sample dataset
from datasets import load_dataset

# TODO: define your dataset and config using the path and name parameters
dataset = load_dataset(path="HuggingFaceTB/smoltalk", name="everyday-conversations")
dataset

DatasetDict({
    train: Dataset({
        features: ['full_topic', 'messages'],
        num_rows: 2260
    })
    test: Dataset({
        features: ['full_topic', 'messages'],
        num_rows: 119
    })
})

## 3. Fine-tune LLM using `trl` and the `SFTTrainer` with LoRA

The [SFTTrainer](https://huggingface.co/docs/trl/sft_trainer) from `trl` provides integration with LoRA adapters through the [PEFT](https://huggingface.co/docs/peft/en/index) library. Key advantages of this setup include:

1. **Memory Efficiency**:
   - Only adapter parameters are stored in GPU memory
   - Base model weights remain frozen and can be loaded in lower precision
   - Enables fine-tuning of large models on consumer GPUs

2. **Training Features**:
   - Native PEFT/LoRA integration with minimal setup
   - Support for QLoRA (Quantized LoRA) for even better memory efficiency

3. **Adapter Management**:
   - Adapter weight saving during checkpoints
   - Features to merge adapters back into base model

We'll use LoRA in our example, which combines LoRA with 4-bit quantization to further reduce memory usage without sacrificing performance. The setup requires just a few configuration steps:
1. Define the LoRA configuration (rank, alpha, dropout)
2. Create the SFTTrainer with PEFT config
3. Train and save the adapter weights


## Using TRL with PEFT ##

PEFT methods can be combined with TRL for fine-tuning to reduce memory requirements. We can pass the LoraConfig to the model when loading it.

In [4]:
from peft import LoraConfig
from transformers import TrainingArguments

peft_config = LoraConfig(
    r=6, # r: rank dimension for LoRa update matrices (smaller = more compression)
    lora_alpha=8, # scaling factor for LoRA layers (higher = stronger adaptation; typically 2x rank)
    lora_dropout=0.05, # dropout probability for LoRA layers (helps prevent overfitting)
    bias="none", # Bias type for LoRA. the corresponding biases will be updated during training.
    task_type="CAUSAL_LM", # Task type for model architecture

)

In [26]:
from trl import SFTTrainer, SFTConfig

# Define SFTConfig for trainer arguments, adapting from previous TrainingArguments
sft_args = SFTConfig(
    output_dir="pop123-ux/ybelkada/opt-350m-lora",
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    push_to_hub=True,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    max_grad_norm=0.3,
    warmup_steps=5, # Changed from warmup_ratio to warmup_steps
    lr_scheduler_type="constant",
    logging_steps=10,
    bf16=False, # Changed to False as bf16 is not supported
    fp16=True,  # Explicitly enable fp16 if bf16 is not supported
    report_to="none",
)

tokenizer.chat_template = "{% for message in messages %}{{'<|' + message['role'] + '|>\n' + message['content'] + '</s>\n'}}{% endfor %}"

# Create SFTTrainer with LoRA configuration
trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    processing_class=tokenizer
)


Tokenizing train dataset:   0%|          | 0/2260 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2260 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2260 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/119 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/119 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/119 [00:00<?, ? examples/s]

## Merging LoRA Adapters ##

After training with LoRA, we might want to merge the adapter weights back into the base model for easier deployment. This creates a single model with the combined weights, eliminating the need to load adapters separately during inference

The merging process requires attention to memory management and precision. Since we need to load both the base model and adapter weights simultaneously, we need to ensure sufficient GPU/CPU memory is available. Using device_map='auto' in transformers we find the correct device for the model based on our available hardware.

We must maintain consistent precision (e.g., float16) throughout the process, matching the precision used during training and saving the merged model in the same format for deployment.

## Merging Implementation ##

After training a LoRA adapter, we can merge the adapter weights back into the base model. Here's how to do it:

In [ ]:
import torch
from transformers import AutoModelForCausalML
from peft import PeftModel

# Load the base model
base_model = AutoModelForCausalLM.from_pretrained(
    'base_model_name', torch_dtype=torch.float16, device_map='auto'
)

# Load the PEFT model with adapter
peft_model = PeftModel.from_pretrained(
    base_model, "path/to/adapter", torch_dtype=torch.float16
)

# Merge the adapter weights with the base model
merged_model = peft_model.merge_and_unload()

# Save the merged model
merged_model.save_pretrained("path/to/save/merged_model")

If we encounter size discrepancies in the saved model, we must ensure we're also saving the tokenizer:

In [ ]:
# Save both model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("base_model_name")
merged_model.save_pretrained("path/to/save/merged_model")
tokenizer.save_pretrained("path/to/save/merged_model")

<div style='background-color: lightblue; padding: 10px; border-radius: 5px; margin-bottom: 20px; color:black'>
    <h2 style='margin: 0;color:blue'>Bonus Exercise: Load LoRA Adapter</h2>
    <p>Use what you learnt from the example note book to load your trained LoRA adapter for inference.</p>
</div>

In [29]:
# free the memory again
import torch

del model
del trainer
torch.cuda.empty_cache()

NameError: name 'trainer' is not defined

In [ ]:
import torch
from peft import AutoPeftModelForCausalLM # Kept for potential future use or context
from transformers import AutoTokenizer, pipeline, AutoModelForCausalLM

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

# Load Model with PEFT adapter (changed to AutoModelForCausalLM for base model)
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
model = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B", device_map="auto", dtype=torch.float16
)
model.to(device)

pipe = pipeline(
    "text-generation", model=model, tokenizer=tokenizer, device=device
)

Lets test some prompt samples and see how the model performs.

In [ ]:
prompts = [
    "What is the capital of Germany? Explain why thats the case and if it was different in the past?",
    "Write a Python function to calculate the factorial of a number.",
    "A rectangular garden has a length of 25 feet and a width of 15 feet. If you want to build a fence around the entire garden, how many feet of fencing will you need?",
    "What is the difference between a fruit and a vegetable? Give examples of each.",
]


def test_inference(prompt):
    prompt = pipe.tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    outputs = pipe(
        prompt,
    )
    return outputs[0]["generated_text"][len(prompt) :].strip()


for prompt in prompts:
    print(f"    prompt:\n{prompt}")
    print(f"    response:\n{test_inference(prompt)}")
    print("-" * 50)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    prompt:
What is the capital of Germany? Explain why thats the case and if it was different in the past?


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    response:
Okay, so I need to figure out the capital of Germany and explain why it is that way. I remember hearing that it's something like Berlin, but I'm not exactly sure. Let me think. Germany is a country with a long history, so maybe it's a big city. I know that in Europe, capitals are often big cities, so that makes sense.

Why is Berlin the capital? Well, first of all, Berlin is located in the Brandenburg Gate, which is a major landmark in Germany. It's a big city, so that probably plays a role. Maybe Berlin has a lot of attractions or landmarks that other cities don't. I think Berlin has a lot of universities, especially technical universities, which might be a big reason. Also, Berlin is a major center for business, finance, and technology. So that probably attracts a lot of people.

I also remember that Berlin has a rich history. It was originally the capital of the Prussian Empire, but during the Napoleonic Wars, it became a significant center for the French. So that make